# Week 05 — Missing labels and conversion modeling

**Goal.** Rebuild, on open data, the thing you owned at Google: estimating conversions that were never observed because of consent and signal loss.

**Deliverable.** A two-model correction (observed CVR + gap model) compared against naive and oracle, written up as an explainer.

**Rough shape of the week.** 2h reading (PU learning) · 5h building · 2h write-up — this one is the blog post.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The setup

Split users into a *consented* segment whose conversions you observe and a *consentless*
segment whose conversions are invisible. You keep the ground truth aside so you can
score yourself — which is the one luxury this simulation has and the real problem does
not.

Three models to compare:

| model | trained on | what it represents |
|---|---|---|
| naive | observed labels, all traffic | what you get if you ignore the problem |
| corrected | observed labels + a model of the gap | conversion modeling |
| oracle | true labels | the ceiling; unattainable in production |

The gap between **naive and oracle** is the size of the business problem. The gap between
**corrected and oracle** is how much of it you recovered. Report both as one number each
and the week has landed.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())
print(f"{len(df):,} rows, {df.timestamp.max()/86400:.1f} days")

sp = split.time_split(df, "timestamp", train_frac=0.7, val_frac=0.1)
split.check_no_leakage(df, sp)
print(sp)

train, val, test = sp.apply(df)

## 1. Simulate the loss

Drop conversions for ~30% of users. Do it **by user, not by row** — consent is a property
of a person, and dropping rows at random creates a much easier, and fake, problem.

Then make it harder and more realistic: make consent *non-random*. If consent correlates
with a feature that also predicts conversion (mobile users consent less and convert
less, say), the missingness is MNAR and a naive reweighting will not save you. Do the
MCAR version first to get the pipeline working, then the MNAR version to get the finding.

In [ ]:
rng = np.random.default_rng(0)
users = df.uid.unique()
consentless = set(rng.choice(users, size=int(0.3 * len(users)), replace=False))

df["consented"] = (~df.uid.isin(consentless)).astype(int)
df["y_observed"] = df.conversion * df.consented   # invisible conversions become zeros

print(f"true CVR={df.conversion.mean():.4%}   observed CVR={df.y_observed.mean():.4%}")
print(f"-> {1 - df.y_observed.sum()/df.conversion.sum():.1%} of conversions are invisible")

## 2. Naive model

Train on `y_observed` across all traffic. Evaluate on the true label. Note what happens
to `calibration_ratio` — it should land near 0.7, i.e. you would systematically underbid
by 30%, which in a real account means losing the auctions you should have won and slowly
starving the campaign.

In [ ]:
# TODO

## 3. The correction

This is the interesting part, and there is more than one defensible design. Try at least
two and argue for one:

**(a) Scale-up.** Estimate the observation rate $r$ (share of conversions you see) and
divide. Trivial, and correct only if consent is independent of everything. Establish it
as the baseline for the *correction*, not just for the model.

**(b) Two-model / imputation.** Train the CVR model on the consented segment only, where
labels are clean. Apply it to consentless traffic to *impute* expected conversions. This
is closest to how modeled conversions actually work. The catch, and the thing to write
about: if consent is MNAR, the consented segment is a biased sample and the model you
transfer is fitted to the wrong population.

**(c) PU learning.** Treat it formally: positives are reliable, "negatives" are
unlabelled. `papers/kiryo2017-nnpu.pdf` gives you a non-negative risk estimator that does
not collapse. This is the principled version of (b).

In [ ]:
# TODO: implement at least (a) and (b); (c) if the week allows

## 4. Oracle and the scoreboard

Same architecture, true labels, upper bound.

Then the table. And the sentence that matters: *"conversion modeling recovered X% of the
conversions that privacy loss made invisible, at the cost of Y in AUC."* If you cannot
fill in X and Y, the week is not finished.

In [ ]:
# rows = {}
# for name, p in [("naive", p_naive), ("scale_up", p_scaled), ("two_model", p_imputed), ("oracle", p_oracle)]:
#     rows[name] = metrics.evaluate(y_true_test, p)
# pd.DataFrame(rows).T

## 5. Where it breaks

The honest failure mode, and the part an interviewer will push on: sweep the consentless
share from 10% to 70% and plot recovery quality against it. There is a point where the
consented segment is too small or too unrepresentative to transfer from. Find it, name
it, and say what you would do past it (aggregate-only measurement — which is Week 10).

In [ ]:
# for share in [0.1, 0.2, 0.3, 0.5, 0.7]:
#     ... -> plot recovery vs share
# print(plots.save(fig, 5, "recovery_vs_consent_loss"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=5,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=5))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week05_* results/
git commit -m "week 05: <the finding, not the task>"
```